In [14]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, Embedding, LSTM, Concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Configuration
IMAGE_SIZE = (224, 224)
MAX_LEN = 100
VOCAB_SIZE = 5000
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
CSV_FILE = "symptoms.csv"
IMAGE_FOLDER = r"C:\Users\User\Downloads\Skin-disease-dataset\train"
MODEL_SAVE_PATH = "best_multimodal_model.h5"
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.1

print("Libraries imported and configuration set!")

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [15]:
# Cell 2: Load and Initial Data Processing
print("Loading CSV data...")

# Load CSV
df = pd.read_csv(CSV_FILE)
print(f"Original dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:\n{df.head()}")

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Remove rows with missing values
df.dropna(inplace=True)
print(f"Dataset shape after removing NaN: {df.shape}")

# Check unique labels
print(f"Unique labels: {df['label'].unique()}")
print(f"Number of unique labels: {df['label'].nunique()}")

# Show label distribution
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

# ===============================================

Loading CSV data...


NameError: name 'CSV_FILE' is not defined

In [ ]:
# Cell 3: Label Encoding and Text Tokenization
print("Encoding labels and tokenizing text...")

# Label encoding
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print(f"Label mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{i}: {label}")

# Tokenize symptoms
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["symptom"])

# Convert symptoms to sequences
sequences = tokenizer.texts_to_sequences(df["symptom"])
df["token_ids"] = [seq for seq in sequences]

print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Sample tokenized sequence: {sequences[0]}")
print(f"Original text: {df['symptom'].iloc[0]}")

# ===============================================

In [ ]:
# Cell 4: Create Image Paths
print("Creating image paths...")

def create_balanced_image_paths(df, image_folder):
    """Create balanced image paths for each label"""
    image_paths = []
    
    for idx, row in df.iterrows():
        label = row["label"]
        label_dir = os.path.join(image_folder, label)
        
        if not os.path.isdir(label_dir):
            print(f"Warning: Folder not found for label: {label}")
            image_paths.append(None)
            continue
            
        image_list = [f for f in os.listdir(label_dir) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        if not image_list:
            print(f"Warning: No images found in folder: {label_dir}")
            image_paths.append(None)
            continue
            
        # Use modulo for balanced selection
        img_idx = idx % len(image_list)
        img_path = os.path.join(label_dir, image_list[img_idx])
        image_paths.append(img_path)
        
    return image_paths

# Create image paths
image_paths = create_balanced_image_paths(df, IMAGE_FOLDER)
df["image_path"] = image_paths

# Remove samples with missing images
before_count = len(df)
df = df[df["image_path"].notna()].reset_index(drop=True)
after_count = len(df)

print(f"Samples before removing missing images: {before_count}")
print(f"Samples after removing missing images: {after_count}")
print(f"Final dataset shape: {df.shape}")

# ===============================================

In [ ]:
# Cell 5: Data Splitting
print("Splitting data into train/validation/test sets...")

# Split data
train_df, temp_df = train_test_split(
    df, test_size=VALIDATION_SPLIT + TEST_SPLIT, 
    stratify=df['label_id'], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=TEST_SPLIT/(VALIDATION_SPLIT + TEST_SPLIT),
    stratify=temp_df['label_id'], random_state=42
)

print(f"Data split:")
print(f"Train: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples") 
print(f"Test: {len(test_df)} samples")

# Check label distribution in each set
print(f"\nTrain set label distribution:\n{train_df['label'].value_counts()}")
print(f"\nValidation set label distribution:\n{val_df['label'].value_counts()}")
print(f"\nTest set label distribution:\n{test_df['label'].value_counts()}")

# ===============================================

✅ Progress saved to: multimodal_dataset.csv


In [ ]:
# Cell 6: Data Preprocessing Functions
def preprocess_function(image_path, tokens, label):
    """Preprocess image and text data"""
    try:
        # Load and preprocess image
        image = tf.io.read_file(image_path)
        image = tf.image.decode_image(image, channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        
        # Ensure proper shapes
        image = tf.reshape(image, (*IMAGE_SIZE, 3))
        
        return image, tokens, label
        
    except Exception as e:
        # Return placeholder data for invalid samples
        print(f"Error processing image: {e}")
        return (tf.zeros((*IMAGE_SIZE, 3), dtype=tf.float32),
               tf.zeros([MAX_LEN], dtype=tf.int32),
               tf.constant(-1, dtype=tf.int32))

def tf_preprocess_wrapper(data):
    """TensorFlow wrapper for preprocessing function"""
    image, tokens, label = tf.py_function(
        preprocess_function,
        [data["image_path"], data["tokens"], data["labels"]],
        [tf.float32, tf.int32, tf.int32]
    )
    
    # Set shapes
    image.set_shape((*IMAGE_SIZE, 3))
    tokens.set_shape([MAX_LEN])
    label.set_shape([])
    
    return {"image_input": image, "text_input": tokens}, label

def create_tf_dataset(df, is_training=True):
    """Create TensorFlow dataset with proper preprocessing"""
    print(f"Creating {'training' if is_training else 'validation/test'} dataset...")
    
    # Pad sequences
    padded_sequences = pad_sequences(
        df["token_ids"].tolist(), 
        maxlen=MAX_LEN, 
        padding='post'
    )
    
    dataset = tf.data.Dataset.from_tensor_slices({
        "image_path": df["image_path"].values,
        "tokens": padded_sequences,
        "labels": df["label_id"].values
    })
    
    # Map preprocessing function
    dataset = dataset.map(
        tf_preprocess_wrapper,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Filter out invalid samples
    dataset = dataset.filter(lambda x, y: tf.not_equal(y, -1))
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)
        
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

print("Preprocessing functions defined!")

# ===============================================

In [ ]:
# Cell 7: Create TensorFlow Datasets
print("Creating TensorFlow datasets...")

# Create datasets
train_dataset = create_tf_dataset(train_df, is_training=True)
val_dataset = create_tf_dataset(val_df, is_training=False)
test_dataset = create_tf_dataset(test_df, is_training=False)

# Test dataset creation
print("Testing dataset creation...")
for data, labels in train_dataset.take(1):
    print(f"Image batch shape: {data['image_input'].shape}")
    print(f"Text batch shape: {data['text_input'].shape}")
    print(f"Labels batch shape: {labels.shape}")
    break

print("Datasets created successfully!")

# ===============================================

: 

In [ ]:
def train(self, train_dataset, val_dataset):
        """Train the model with callbacks"""
        print("Starting training...")
        
        # Callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_accuracy',
                patience=10,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            ),
            ModelCheckpoint(
                self.config.MODEL_SAVE_PATH,
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            )
        ]
        
        # Train model
        history = self.model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=self.config.EPOCHS,
            callbacks=callbacks,
            verbose=1
        )
        
        return history

❌ Error processing: C:\Users\User\Downloads\Skin-disease-dataset\train\Acne\pigmentation_0_1889.jpeg, Small red, tender bumps called papules that can get worse.
   ⚠️ Exception: Failed copying input tensor from /job:localhost/replica:0/task:0/device:GPU:0 to /job:localhost/replica:0/task:0/device:CPU:0 in order to run ResizeBilinear: Dst tensor is not initialized. [Op:ResizeBilinear]
❌ Error processing: C:\Users\User\Downloads\Skin-disease-dataset\train\Acne\aug_3761_acne-histology-5.jpg, Small red, tender bumps called papules
   ⚠️ Exception: {{function_node __wrapped__DecodeJpeg_device_/job:localhost/replica:0/task:0/device:CPU:0}} OOM when allocating tensor with shape[488,720,3] and type uint8 on /job:localhost/replica:0/task:0/device:CPU:0 by allocator cpu [Op:DecodeJpeg]
❌ Error processing: C:\Users\User\Downloads\Skin-disease-dataset\train\Acne\aug_128_hidradenitis-suppurativa-64.jpg, Small red, tender bumps called papules mostly visible.
   ⚠️ Exception: Failed copying input ten

In [ ]:
# Cell 8: Build Model Architecture (FIXED)
def build_multimodal_model(num_classes):
    """Build the multimodal model"""
    print("Building multimodal model...")
    
    # Image branch - MobileNetV2
    img_input = Input(shape=(*IMAGE_SIZE, 3), name="image_input")
    cnn_base = MobileNetV2(
        include_top=False, 
        weights='imagenet', 
        input_tensor=img_input
    )
    
    # Fine-tune last few layers
    for layer in cnn_base.layers[:-20]:
        layer.trainable = False
    for layer in cnn_base.layers[-20:]:
        layer.trainable = True
        
    x1 = GlobalAveragePooling2D()(cnn_base.output)
    x1 = BatchNormalization()(x1)
    x1 = Dense(256, activation='relu')(x1)
    x1 = Dropout(0.3)(x1)
    
    # Text branch - LSTM
    txt_input = Input(shape=(MAX_LEN,), name="text_input")
    x2 = Embedding(
        input_dim=VOCAB_SIZE, 
        output_dim=128, 
        mask_zero=True
    )(txt_input)
    x2 = LSTM(128, dropout=0.2, recurrent_dropout=0.2)(x2)
    x2 = BatchNormalization()(x2)
    x2 = Dense(128, activation='relu')(x2)
    x2 = Dropout(0.3)(x2)
    
    # Fusion layer
    merged = Concatenate()([x1, x2])
    x = Dense(256, activation='relu')(merged)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    # Output layer
    output = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    # Create model
    model = Model(inputs=[img_input, txt_input], outputs=output)
    
    # Compile with custom optimizer
    optimizer = Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"Model built with {num_classes} output classes")
    return model

# Build the model - PASS num_classes as parameter
try:
    num_classes = len(label_encoder.classes_)
    print(f"Number of classes: {num_classes}")
    model = build_multimodal_model(num_classes)
    print("Model built successfully!")
    model.summary()
except Exception as e:
    print(f"Error building model: {e}")
    print("Please check if label_encoder is properly defined in previous cells")

# Alternative: If label_encoder is not defined, you can define num_classes manually
# For example, if you know you have 5 disease classes:
# num_classes = 5  # Replace with your actual number of classes
# model = build_multimodal_model(num_classes)

ValueError: Invalid `predicate`. `predicate` must return a `tf.bool` scalar tensor, but its return type is TensorSpec(shape=(None,), dtype=tf.bool, name=None).

In [ ]:
# Cell 9: Setup Training Callbacks
print("Setting up training callbacks...")

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configured!")

# ===============================================

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 112, 112, 32  864         ['image_input[0][0]']            
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 112, 112, 32  128         ['Conv1[0][0]']                  
                                )                                                           

In [ ]:
# Cell 10: Train the Model
print("Starting model training...")

# Train model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("Training completed!")

# ===============================================


Starting model training...


NameError: name 'model' is not defined

In [ ]:
# Cell 11: Plot Training History
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# ===============================================

In [ ]:
# Cell 12: Model Evaluation
print("Evaluating model on test set...")

# Predictions on test set
predictions = model.predict(test_dataset)
y_pred = np.argmax(predictions, axis=1)
y_true = test_df['label_id'].values

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true, y_pred, 
    target_names=label_encoder.classes_
))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# ===============================================

In [ ]:
# Cell 13: Save Model Components
print("Saving model components...")

# Create directory for saving components
os.makedirs("model_components", exist_ok=True)

# Save tokenizer
with open("model_components/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open("model_components/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# Save model (already saved by ModelCheckpoint callback)
print(f"Model saved to: {MODEL_SAVE_PATH}")
print("Tokenizer and label encoder saved to: model_components/")

# ===============================================

In [ ]:
# Cell 14: Single Prediction Function
def predict_single_sample(image_path=None, text="", confidence_threshold=0.5):
    """Make prediction on single sample"""
    
    if image_path is None:
        image = np.zeros((*IMAGE_SIZE, 3))
    else:
        try:
            img = Image.open(image_path).convert("RGB")
            img = img.resize(IMAGE_SIZE)
            image = np.array(img) / 255.0
        except Exception as e:
            print(f"Error loading image: {e}")
            image = np.zeros((*IMAGE_SIZE, 3))
            
    image = np.expand_dims(image, axis=0)
    
    # Process text
    if not text:
        text = "no symptoms described"
        
    tokens = tokenizer.texts_to_sequences([text])
    tokens = pad_sequences(tokens, maxlen=MAX_LEN)
    
    # Predict
    predictions = model.predict({
        "image_input": image, 
        "text_input": tokens
    }, verbose=0)
    
    # Get top predictions
    pred_probs = predictions[0]
    top_indices = np.argsort(pred_probs)[::-1][:3]
    
    results = []
    for idx in top_indices:
        confidence = pred_probs[idx]
        if confidence >= confidence_threshold:
            results.append({
                'disease': label_encoder.classes_[idx],
                'confidence': float(confidence)
            })
    
    return results if results else [{'disease': 'uncertain', 'confidence': 0.0}]

print("Single prediction function defined!")

# ===============================================

In [ ]:
# Cell 15: Test Single Prediction
print("Testing single prediction...")

# Test with placeholder data (no image)
test_results = predict_single_sample(
    image_path=None,  # Will use placeholder image
    text="red itchy bumps on skin with inflammation"
)

print(f"Test prediction results: {test_results}")

# If you have a specific image to test, uncomment and modify:
# test_results_with_image = predict_single_sample(
#     image_path="path/to/your/test/image.jpg",
#     text="describe your symptoms here"
# )
# print(f"Prediction with image: {test_results_with_image}")

print("All cells completed successfully!")